# judge 프롬프트와 편향

Python 3.12 새 커널에서 후보 순서를 바꾸는 위치 편향 실험부터 확인합니다.

## 개념을 코드에 연결하기

**호출 표기와 실행 시점.** LLM judge에는 평가 기준, 입력 필드의 역할, 점수 범위, 출력 형식을 명시합니다. 같은 두 후보의 순서만 바꿔 점수가 달라지는지 비교하면 위치 편향을 관찰할 수 있습니다. 모델 이름·프롬프트 해시·후보 순서를 함께 저장해야 나중에 결과를 재현할 수 있습니다.

**다른 상황에 적용.** 두 요약문을 비교할 때 A/B 순서를 바꿔 두 번 평가하고 선택이 뒤집히면 사람 검토 대상으로 보낼 수 있습니다. 평가 대상의 이름이나 길이가 판단에 영향을 주는지도 조건 하나씩 바꿔 확인합니다.

**비교와 경계.** 프롬프트를 고정해도 judge가 객관적 정답이 되는 것은 아닙니다. 위치 편향이 없었다는 한 실험도 다른 입력의 편향 부재를 보장하지 않습니다. 코드로 검사할 수 있는 형식·숫자는 코드 평가자를 우선합니다.


## Python 3.12 패키지 버전 고정

**이 셀의 코드 읽기.** `%pip install`은 이 노트북 커널이 사용할 패키지를 설치합니다. 앞 셀의 값을 입력으로 사용하지 않으며, 설치가 끝난 뒤의 코드 셀에서 패키지를 가져와 사용합니다.


In [1]:
# 실습 환경에서 사용하는 보조 패키지 버전을 고정합니다.
%pip install -q python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## 후보 순서를 바꾸는 위치 편향 실험

**이 셀의 코드 읽기.** `biased_pairwise(first: tuple[str, str], second: tuple[str, str])`는 함수를 정의하는 표기입니다. 이 셀에서는 함수 객체만 준비되고 들여쓴 본문은 뒤에서 함수를 호출할 때 실행됩니다. `return` 뒤의 값은 화면 출력이 아니라 호출한 코드로 전달되는 반환값입니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [2]:
# 첫 후보만 고르는 잘못된 판정기로 위치 편향이 생기는 구조를 재현합니다.
short = ('short', '도서는 7권까지 14일 동안 빌릴 수 있습니다. [E01]')
long = ('long', short[1] + ' 도서관은 시민을 위한 지식 공간입니다.')
def biased_pairwise(first: tuple[str, str], second: tuple[str, str]) -> str:
    return first[0]  # 잘못된 평가자: 내용 대신 첫 후보 ID를 선택
winner_ab = biased_pairwise(short, long)
winner_ba = biased_pairwise(long, short)
print('short/long 승자:', winner_ab)
print('long/short 승자:', winner_ba)
print('순서를 바꿔도 같은 후보인가:', winner_ab == winner_ba)

short/long 승자: short
long/short 승자: long
순서를 바꿔도 같은 후보인가: False


## 평가 모델 사양과 프롬프트 해시 고정

**이 셀의 코드 읽기.** `sha256(bytes)`는 같은 입력인지 비교할 해시 객체를 만들며 원문을 복원하는 암호화 함수가 아닙니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [3]:
# judge 모델·온도·프롬프트 해시·출력 형식을 하나의 사양으로 묶습니다.
import hashlib
judge_prompt = '근거로 뒷받침되면 1, 아니면 0. JSON만 출력.'
spec = {
 'model': 'gpt-5.4-mini', 'temperature': 0,
 'prompt_version': 'groundedness-v1.0.0',
 'prompt_sha256': hashlib.sha256(judge_prompt.encode()).hexdigest(),
 'output_schema': {'score': '0|1', 'reason': 'string'},
}
print(spec)

{'model': 'gpt-5.4-mini', 'temperature': 0, 'prompt_version': 'groundedness-v1.0.0', 'prompt_sha256': 'e8b76f93d743595c23e7812d6d5b26a85b4bc1ee407dfa89313850d0261793e7', 'output_schema': {'score': '0|1', 'reason': 'string'}}


## 저장된 실제 평가 모델 버전 확인

**이 셀의 코드 읽기.** `json.loads(text)`는 JSON 문자열을 파이썬 딕셔너리나 목록으로 변환해 반환합니다. `read_text()`는 파일 내용을 문자열로 반환합니다. `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다. `assert`는 조건이 참이면 출력 없이 통과하고 거짓이면 실행을 중단하므로, 바로 앞 결과를 검증하는 역할을 합니다.


In [4]:
# 세 사례의 실제 judge 기록과 운영 모델 미정 상태를 확인합니다.
from pathlib import Path
import json
root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'labs/part4/ch15-evaluation-advanced').is_dir())
data = json.loads((root / 'labs/part4/ch15-evaluation-advanced/data/evaluation-cases.json').read_text())
for case in data['cases']:
    print(case['qid'], case['judge_model'], case['judge_prompt_version'])
assert all(x['judge_placeholder'] == '[[미정: judge 모델]]' for x in data['cases'])

Q002 gpt-5.4-mini groundedness-v1.0.0
Q006 gpt-5.4-mini groundedness-v1.0.0
Q090 gpt-5.4-mini groundedness-v1.0.0
